<a href="https://colab.research.google.com/github/noursalloum5/News-Rag/blob/main/News_Rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **News RAG Project**



## Setup & Dependencies


In [ ]:
!pip install --upgrade langchain langchain_community langchain-openai langchain-pinecone pinecone-client unstructured sentence-transformers transformers accelerate


In [ ]:
import requests
import os
import re
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from pinecone import Pinecone
from pinecone import Pinecone, ServerlessSpec
from pinecone.exceptions import PineconeApiException
from langchain_pinecone import PineconeVectorStore
from transformers import pipeline
from huggingface_hub import login
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_community.llms import HuggingFacePipeline
import gradio as gr
from google.colab import userdata

## Data Collection & Preprocessing


In [ ]:
API_KEY = userdata.get('NEWS_API_KEY')
folder = "news"

os.makedirs(folder, exist_ok=True)

url = f"https://newsapi.org/v2/everything?q=world&language=en&pageSize=40&sortBy=publishedAt&apiKey={API_KEY}"
response = requests.get(url)
data = response.json()
articles = data.get("articles", [])


In [ ]:
def clean_text(text):
    if not text:
        return ""
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"http\S+", "", text)
    return text.strip()


In [ ]:
for i, article in enumerate(articles):
    title = clean_text(article.get("title", ""))
    description = clean_text(article.get("description", ""))
    content = clean_text(article.get("content", ""))
    author = article.get("author", "Unknown Author")
    source = article.get("source", {}).get("name", "Unknown Source")
    published_at = article.get("publishedAt", "Unknown Date")

    if not (title or description or content):
        continue

    text = (
        f"Title: {title}\n"
        f"Author: {author}\n"
        f"Source: {source}\n"
        f"Published At: {published_at}\n\n"
        f"{description}\n\n{content}"
    )

    filename = os.path.join(folder, f"news_{i}.txt")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(text)
print(f"Saved {len(data.get('articles', []))} news articles in '{folder}'")


## Document Preparation


In [ ]:
loader = DirectoryLoader('./news/')
docs_before_split = loader.load()

In [ ]:
docs_before_split[0]

In [ ]:
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=50
)
docs_after_split=text_splitter.split_documents(docs_before_split)

In [ ]:
len(docs_after_split[0].page_content)

## Embeddings & Vector Store


In [ ]:
embeddings = HuggingFaceBgeEmbeddings(
    model_name= "sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs = {'device' : 'cpu'},
    encode_kwargs = {'normalize_embeddings' : True}
)

In [ ]:
pinecone_api_key=userdata.get('pinecone_api_key')
pc=Pinecone(api_key=pinecone_api_key)
os.environ["PINECONE_API_KEY"] = pinecone_api_key

In [ ]:
index_name = "news-rag-bot"

try:

    if index_name not in pc.list_indexes():

        pc.create_index(
            name=index_name,
            dimension=384,
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1")
        )
        print(f"Index '{index_name}' created.")
    else:
        print(f"Index '{index_name}' already exists.")

    index = pc.Index(index_name)

except PineconeApiException as e:

    if e.status == 409:
        print(f"Index '{index_name}' already exists. Connecting to the existing index.")
        index = pc.Index(index_name)
    else:

        raise e

In [ ]:
vector_store = PineconeVectorStore.from_documents(
    docs_after_split,
    embeddings,
    index_name=index_name
)

In [ ]:
query="What is happening now in Gaza?"

In [ ]:
relevant_documents = vector_store.similarity_search(query)

In [ ]:
relevant_documents

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity" , search_kwargs={"k" : 3})

## RAG Pipeline


In [ ]:
login(token=userdata.get('hf_token'))

In [ ]:
pipe = pipeline(
    "text-generation",
    model="meta-llama/Llama-3.2-3B-Instruct",
    device_map="auto",
    torch_dtype="auto",
    max_new_tokens=300,
    temperature=0.1,
    return_full_text=False
)

In [ ]:
prompt_template = """
You are a highly accurate news assistant. Follow these rules strictly:

1. Answer ONLY using information explicitly stated in the provided context.
2. If the answer is not directly stated in the context, reply exactly:
   "I do not have enough information to answer this."
3. Do NOT infer, guess, suggest, or assume anything not literally written.
4. Do NOT say “this suggests”, “it implies”, or provide interpretations.
5. Be concise and factual.
6. Only mention author, source, or publication date if the user specifically asks for them.
7. Do not repeat the context.

Context:
{context}

Question:
{question}

Answer:
"""


PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)




PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)


In [ ]:
llm = HuggingFacePipeline(pipeline=pipe, model_kwargs={})

In [ ]:
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | PROMPT
    | llm
)


## Gradio


In [ ]:
def chat_with_rag(history, new_message):
    if not new_message.strip():
        return history, ""

    # Call the new RAG chain instead of retrievalQA
    answer = rag_chain.invoke(new_message)

    # Add to chat history
    history = history + [(new_message, answer)]
    return history, ""



# Gradio chat app
def gradio_chat_app():
    with gr.Blocks() as app:
        gr.Markdown("# RAG Model Chat Interface")
        gr.Markdown("Chat with your knowledge retriever in a conversational format.")

        chatbot = gr.Chatbot(label="Chat Interface")
        user_input = gr.Textbox(label="Your message", placeholder="Type something...", lines=1)
        send_button = gr.Button("Send")
        clear_button = gr.Button("Clear Chat")

        # Send button: updates chat history
        send_button.click(
            fn=chat_with_rag,
            inputs=[chatbot, user_input],
            outputs=[chatbot, user_input]
        )

        # Submit with Enter key: updates chat history
        user_input.submit(
            fn=chat_with_rag,
            inputs=[chatbot, user_input],
            outputs=[chatbot, user_input]
        )

        # Clear button: resets chat
        def clear_chat():
            return [], ""

        clear_button.click(
            fn=clear_chat,
            inputs=[],
            outputs=[chatbot, user_input]
        )

    return app

if __name__ == "__main__":
    app = gradio_chat_app()
    app.launch()


## Conclusion

This project demonstrates a Retrieval-Augmented Generation (RAG) system applied to news articles.
The system retrieves relevant documents using embeddings and a vector database, then generates answers using an LLM.
Overall, the pipeline works well for information retrieval and question answering.